Install Dependencies

In [1]:
import pandas as pd
import json
from typing import Dict, Any, List, Optional

Define the Tool Class

In [2]:
class CSVAnalyzerTool:
    """
    Enterprise Tool for loading, inspecting, analyzing, 
    and generating summary statistics on CSV files.
    """
    
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df: Optional[pd.DataFrame] = None

    def load_data(self) -> Dict[str, Any]:
        """Loads CSV data safely with basic error handling."""
        try:
            self.df = pd.read_csv(self.file_path)
            return {
                "status": "success",
                "rows": len(self.df),
                "columns": list(self.df.columns)
            }
        except FileNotFoundError:
            return {"status": "error", "message": f"File not found at {self.file_path}"}
        except Exception as e:
            return {"status": "error", "message": f"Failed to parse CSV: {str(e)}"}

    def summarize(self) -> Dict[str, Any]:
        """Generates quantitative summary statistics and data profile."""
        if self.df is None:
            load_res = self.load_data()
            if load_res["status"] == "error":
                return load_res
                
        try:
            numeric_summary = self.df.describe().to_dict()
            null_counts = self.df.isnull().sum().to_dict()
            
            return {
                "status": "success",
                "total_records": len(self.df),
                "column_types": {col: str(dtype) for col, dtype in self.df.dtypes.items()},
                "null_value_counts": null_counts,
                "numerical_summary": numeric_summary
            }
        except Exception as e:
            return {"status": "error", "message": f"Analysis error: {str(e)}"}

    def filter_and_aggregate(self, group_by_col: str, target_col: str, agg_func: str = "mean") -> Dict[str, Any]:
        """Performs grouping and aggregation on specified columns."""
        if self.df is None:
            self.load_data()
            
        if group_by_col not in self.df.columns or target_col not in self.df.columns:
            return {
                "status": "error", 
                "message": f"Specified columns '{group_by_col}' or '{target_col}' do not exist."
            }
            
        try:
            result = self.df.groupby(group_by_col)[target_col].agg(agg_func).to_dict()
            return {
                "status": "success",
                "group_by": group_by_col,
                "target": target_col,
                "aggregation": agg_func,
                "data": result
            }
        except Exception as e:
            return {"status": "error", "message": f"Aggregation failed: {str(e)}"}

Create Sample Data to Test

In [3]:
import pandas as pd

sample_data = {
    "Region": ["North", "South", "North", "East", "South", "East", "North"],
    "Product": ["Widgets", "Gadgets", "Widgets", "Gadgets", "Widgets", "Gadgets", "Widgets"],
    "Sales": [1200, 850, 1500, 930, 1100, 760, 1400],
    "Units_Sold": [40, 25, 50, 30, 35, 20, 45]
}

df_mock = pd.DataFrame(sample_data)
df_mock.to_csv("sample_sales.csv", index=False)
print("Mock CSV 'sample_sales.csv' created successfully!")

Mock CSV 'sample_sales.csv' created successfully!


Execute & Test the Tool

In [8]:
analyzer = CSVAnalyzerTool("sample_sales.csv")

# 1. Load Data
print("LOAD DATA")
print(json.dumps(analyzer.load_data(), indent=2))



LOAD DATA
{
  "status": "success",
  "rows": 7,
  "columns": [
    "Region",
    "Product",
    "Sales",
    "Units_Sold"
  ]
}


In [9]:
# 2. Summarize Data
print("\nSUMMARY STATS")
print(json.dumps(analyzer.summarize(), indent=2))




SUMMARY STATS
{
  "status": "success",
  "total_records": 7,
  "column_types": {
    "Region": "object",
    "Product": "object",
    "Sales": "int64",
    "Units_Sold": "int64"
  },
  "null_value_counts": {
    "Region": 0,
    "Product": 0,
    "Sales": 0,
    "Units_Sold": 0
  },
  "numerical_summary": {
    "Sales": {
      "count": 7.0,
      "mean": 1105.7142857142858,
      "std": 278.9179773611556,
      "min": 760.0,
      "25%": 890.0,
      "50%": 1100.0,
      "75%": 1300.0,
      "max": 1500.0
    },
    "Units_Sold": {
      "count": 7.0,
      "mean": 35.0,
      "std": 10.801234497346433,
      "min": 20.0,
      "25%": 27.5,
      "50%": 35.0,
      "75%": 42.5,
      "max": 50.0
    }
  }
}


In [10]:
# 3. Test Aggregation (Average sales per Region)
print("\nAGGREGATION (Mean Sales by Region)")
agg_result = analyzer.filter_and_aggregate(
    group_by_col="Region", 
    target_col="Sales", 
    agg_func="mean"
)
print(json.dumps(agg_result, indent=2))


AGGREGATION (Mean Sales by Region)
{
  "status": "success",
  "group_by": "Region",
  "target": "Sales",
  "aggregation": "mean",
  "data": {
    "East": 845.0,
    "North": 1366.6666666666667,
    "South": 975.0
  }
}
